# Student Performance Analysis

## Objective
The goal of this project is to analyze the factors affecting student performance in exams and build predictive models for math scores.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error,mean_squared_error
from sklearn.tree import DecisionTreeRegressor

## Loading the Dataset
We import the dataset and inspect its structure.

In [ ]:
df=pd.read_csv("Data/StudentsPerformance.csv")
df.head()

## Basic Exploratory Data Analysis
We examine dataset shape, columns, and summary statistics.

In [ ]:
# Data Description
df.shape
df.info()
df.describe()
df.columns
# Any null Values
df.isnull().sum()


## Key Insights

1. Female students perform better in reading and writing.
2. Male students perform slightly better in math.
3. Students who completed the test preparation course tend to score higher.
4. Math has the lowest average score among the three subjects.
5. Reading and writing scores show strong correlation.

In [ ]:
# Which subject has the highest average score?
df[["math score","reading score","writing score"]].mean()


In [ ]:
# What are the most common math scores?
df["math score"].value_counts().head()


In [ ]:
# Do males or females score higher on average?
df.groupby("gender")[["math score","reading score","writing score"]].mean()


In [ ]:
# Do students who completed test preparation perform better?
df.groupby("test preparation course")[["math score","reading score","writing score"]].mean()


In [ ]:
# Does parent education level affect student scores?
df.groupby("parental level of education")[["math score","reading score","writing score"]].mean()

## Data Visualization
We create visualizations to understand patterns in student performance.

In [ ]:
sns.histplot(df["math score"],bins=30)
plt.title("Distribution of Math score")
plt.show()

In [ ]:
sns.boxplot(x='gender',y='math score',data=df)
plt.title("Distribution of Math score by gender")
plt.show()

In [ ]:
sns.scatterplot(x='writing score',y='reading score',data=df)

In [ ]:
sns.barplot(x="parental level of education",y='math score',data=df)
plt.xticks(rotation=45)
plt.show()

In [ ]:
corr=df[["math score","reading score","writing score"]].corr()
corr

In [ ]:
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Correlation Between Subject Scores")
plt.yticks(rotation=360)
plt.show()

##  Model Preparation

We separate the dataset into:
- Features (X): All input variables
- Target (y): Math score (what we want to predict)

In [ ]:
X=df.drop('math score',axis=1)
y=df['math score']

### Encoding Categorical Variables

We convert categorical variables into numerical format using One-Hot Encoding.

In [ ]:
X=pd.get_dummies(X,drop_first=True)

###  Train-Test Split

We split the data into:
- Training set (80%)
- Testing set (20%)

This helps evaluate model performance on unseen data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape)
print(X_test.shape)

## Linear Regression Model

We train a Linear Regression model to predict math scores based on input features.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
predictions[:10]

###  Model Evaluation

We evaluate the model using:
- Mean Absolute Error (MAE)
- Mean Squared Error (MSE)

In [ ]:
mae = mean_absolute_error(y_test, predictions)
print(mae)

In [ ]:
mse = mean_squared_error(y_test, predictions)
print(mse)

###  Understanding Coefficients

- Positive coefficient → increases predicted math score
- Negative coefficient → decreases predicted math score

#### Key Observations:
- Features like **test preparation course** have a strong positive impact
- Some categorical variables (like gender) show high coefficients

##### Important:
- These coefficients are influenced by encoding (one-hot encoding)
- They do NOT directly represent real-world importance
- Values may appear large due to scale differences

##### Therefore, coefficients should be interpreted carefully.

In [ ]:
coeff_df = pd.DataFrame(model.coef_, X.columns, columns=["Coefficient"])
coeff_df = coeff_df.sort_values(by="Coefficient", ascending=False)
print(coeff_df.head(10))

## Feature Engineering

We create a new feature:
- **Average Score** = (Reading + Writing) / 2

This helps capture overall academic performance.

In [ ]:
df['average score']=(df['reading score']+df['writing score'])/2

In [ ]:
#Data pipeline
X = df.drop("math score", axis=1)
X = pd.get_dummies(X, drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
mae_new = mean_absolute_error(y_test, predictions)
print("New MAE:", mae_new)

* After adding the average score feature, model performance slightly decreased (compare with previous MAE).

* This suggests whether the new feature adds useful information or redundancy.

In [ ]:
tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)

y_pred_tree = tree.predict(X_test)

mae_tree = mean_absolute_error(y_test, y_pred_tree)

print("Decision Tree MAE:", mae_tree)
print("Train Score:", tree.score(X_train, y_train))
print("Test Score:", tree.score(X_test, y_test))

###  Decision Tree Analysis

- Train Score ≈ 0.99 (very high)
- Test Score ≈ 0.75 (lower)

##### This indicates **overfitting**

The model memorizes training data but performs worse on unseen data.